# L14 · Series temporales

| | |
|---|---|
| **Qué vas a aprender** | Resamplear, ventanas móviles y cruzar husos horarios |
| **Cuánto tiempo** | 75 min |
| **Datasets** | `logistica_envios.csv`, `ventas_2024_este.csv` |

> **Al terminar vas a poder** medir tendencia y anomalías de demanda, y dejar de
> sumar horas de envíos de dos países que no están en la misma hora.

In [ ]:
import sys
from pathlib import Path

# JupyterLab arrancado en la raíz del proyecto deja el cwd ahí.
# Si abres el notebook con el cwd en notebooks/, sube un nivel.
ROOT = Path.cwd()
if not (ROOT / "datos").exists() and (ROOT.parent / "datos").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks"))
DATOS = ROOT / "datos"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from herramientas import (cargar, mostrar, ficha, guardar_csv, guardar_excel,
                           guardar_figura, parsear_importes, parsear_fecha,
                           clave_canonica, detectar_alfabeto, auditar_fusiones,
                           resumen_entrega, perfil_memoria, contar_falsos_nulos)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
plt.style.use("seaborn-v0_8-whitegrid")
pd.__version__

In [ ]:
log = pd.read_csv(DATOS / "logistica_envios.csv", encoding="utf-8")
log["peso_kg"] = parsear_importes(log["peso_kg"])
log["coste_envio"] = parsear_importes(log["coste_envio"])
mostrar(log, 5, "LOGÍSTICA (crudo)")

## El concepto

Una serie temporal es un DataFrame con el tiempo en el **índice**. Todo lo demás
(`.resample`, `.rolling`, `diff`, `pct_change`) son operaciones que necesitan ese
índice para saber qué filas son consecutivas.

Si tu columna de fecha es texto, ninguna de esas funciona. Y si la fecha está bien
pero en la zona horaria equivocada, los agregados salen desplazados una hora y no
te das cuenta.

## Manos a la obra

In [ ]:
# Un solo formato tira la otra mitad. Guardamos el texto para poder mirarlo.
texto_local = log["fecha_envio_local"].copy()
estricto = pd.to_datetime(texto_local, format="%Y-%m-%dT%H:%M:%S", errors="coerce")
print(f"{int(estricto.isna().sum()):,} filas no caben en formato ISO-T")
print("ejemplo de las que fallan:", texto_local[estricto.isna()].iloc[0])
print()
# En pandas 3, to_datetime sin format tampoco mezcla formatos: el segundo queda en NaT.
inferido = pd.to_datetime(texto_local, errors="coerce")
print("nulos si no pasas format:", int(inferido.isna().sum()))

In [ ]:
# format="mixed" (o "ISO8601") parsea los dos. Se aplica al TEXTO original,
# no a la columna que ya convertiste: los NaT no recuerdan la cadena.
log["fecha_envio_local"] = pd.to_datetime(texto_local, format="mixed", errors="coerce")
print("nulos con format='mixed':", int(log["fecha_envio_local"].isna().sum()))
print()
print("Para fechas ambiguas tipo 01/02/2024 sigue haciendo falta format= y decidir")
print("el día. 'mixed' no adivina si el primer número es el día o el mes.")

In [ ]:
# paso 2: poner el tiempo en el indice
log = log.set_index("fecha_envio_local").sort_index()
print("indice:", type(log.index).__name__, "| rango:", log.index.min(), "->", log.index.max())

diario = log["coste_envio"].resample("D").sum().round(2)
mostrar(diario.head(8), 8, "COSTE DE ENVÍO POR DÍA")

Fíjate en `resample("D")`, no en `resample("d")`. En pandas 2.2+ las
letras minúsculas de tipo mensual (`"M"`) están **deprecadas** en favor de las
aliases nuevas: `"ME"` (month-end), `"QE"`, `"YE"`, `"W"`.

In [ ]:
# ventanas mobiles: media de 7 dias, y su ratio con la anterior
media7 = diario.rolling(7, min_periods=1).mean().round(2)
print("media movil de 7 dias:")
mostrar(pd.DataFrame({"real": diario, "media7": media7}).head(8), 8)

### Cruce de husos: el problema real del dataset

`fecha_envio_local` está en hora local de cada país, y `fecha_envio_utc` es la misma
instante en UTC. La pregunta de negocio ("¿cuántos envíos salieron a las 9 de la
mañana?") **depende del huso**: las 9 de Madrid no son las 9 de Estambul.

In [ ]:
log2 = pd.read_csv(DATOS / "logistica_envios.csv", encoding="utf-8")
log2["utc"] = pd.to_datetime(log2["fecha_envio_utc"], format="%Y-%m-%dT%H:%M:%SZ", utc=True)
log2["local"] = pd.to_datetime(log2["fecha_envio_local"], errors="coerce")

# agrupar por HORA UTC: compara a todos los paises en el mismo marco temporal
por_hora_utc = log2.groupby(log2["utc"].dt.hour).size()
print("envios por HORA UTC (mismo marco para todos los paisos):")
print(por_hora_utc.to_string())

Si usaras la hora **local**, cada país tendría su pico en la misma franja
número, y estarías comparando una curva que no significa nada. Al pasar todo a UTC, la
curva comparativa es real.

## ⚠️ La trampa

### `tz_localize` vs `tz_convert`

Son dos operaciones distintas y equivocarse lanza `TypeError`:

- `tz_localize`: le **asigna** una zona a una fecha que es naive (no tiene zona).
- `tz_convert`: **convierte** una fecha que ya tiene zona a otra zona.

Además, un día de cambio de hora tiene 23 o 25 horas. Un `resample("D")` en ese día
no devuelve 24 filas, y por eso las medias móveis se descuadran si no lo sabes.

In [ ]:
naive = pd.to_datetime("2024-03-31 01:30:00")
con_zona = naive.tz_localize("Europe/Madrid")
print("naive localized ->", con_zona)
print("convertido a UTC->", con_zona.tz_convert("UTC"))
print()
try:
    con_zona.tz_localize("Europe/Istanbul")
except TypeError as e:
    print("localizar dos veces -> TypeError:", str(e)[:60])
print("(porque ya tiene zona: hay que usar tz_convert)")

In [ ]:
# freq="H" está deprecado; la forma nueva es "h".
# inclusive="left" para no contar solo la medianoche del día final.
primavera = pd.date_range(
    "2024-03-30", "2024-04-01", freq="h", tz="Europe/Madrid", inclusive="left"
)
otono = pd.date_range(
    "2024-10-26", "2024-10-28", freq="h", tz="Europe/Madrid", inclusive="left"
)
print("primavera (31 de marzo, el reloj salta):")
print(primavera.floor("D").value_counts().sort_index().to_string())
print()
print("otoño (27 de octubre, el reloj vuelve):")
print(otono.floor("D").value_counts().sort_index().to_string())
print()
print("23 horas en marzo, 25 en octubre. El día de al lado sigue teniendo 24.")

## 🧪 Tu turno

- **Ejercicio 1** — Para `ventas_2024_este.csv` (en `DATOS`), saca las ventas mensuales y calcula
  la variación mensual (`pct_change`). ¿Qué meses tienen salto negativo?
- **Ejercicio 2** — Compara la media móvil de 7 y de 30 días en el coste de envío.
  ¿Dónde se separan más? Pista: mira las fechas.
- **Ejercicio 3** — Con `format="%Y-%m-%dT%H:%M:%S"`, ¿cuántos envíos quedan en `NaT`?
  Mira `texto_local` en esas filas y di qué formato tienen. Con `format="mixed"`
  ese número tiene que ser 0.

## 📦 Workbook integrador: demanda y stock

**Contexto.** Eres el responsable de operaciones de un retailer que vende en España y
exporta a Europa. La pregunta del director es: **¿cuándo hay que reponer stock?** Para
responderla necesitas dos cosas: la demanda diaria (para ver la tendencia) y una
detección de anomalías (los días que se disparan las ventas sin motivo).

**Tu trabajo.** La serie es el **coste diario de envío**, no las ventas.

1. Serie diaria de coste de envíos, con media móvil de 7 días.
2. Detección de anomalías sobre las **diferencias** diarias: un salto de más de
   2 desviaciones típicas respecto de la media de los saltos. Una serie con
   tendencia marca los extremos si miras los valores en vez de los saltos.
3. Exportar la serie con la marca de anomalía a `salida/L14_demanda_diaria.csv`.
4. Un gráfico con la serie, la media móvil y las anomalías, en PNG y SVG.

In [ ]:
# 1. serie diaria
serie = log["coste_envio"].resample("D").sum().dropna()
serie.name = "coste_diario"

# 2. deteccion de anomalias sobre las DIFERENCIAS (no sobre los valores)
diferencias = serie.diff()
umbral = diferencias.std() * 2
es_anomalia = (diferencias - diferencias.mean()).abs() > umbral

print(f"dias analizados   : {len(serie)}")
print(f"umbral (2 sigma)  : {umbral:,.2f} € de salto diario")
print(f"dias anómalos     : {es_anomalia.sum()}")
print()
anomalias = serie[es_anomalia.fillna(False)].sort_values(ascending=False)
mostrar(anomalias.head(10), 10, "DÍAS CON SALTO ANÓMALO")

In [ ]:
# 3. el entregable
informe = pd.DataFrame({
    "coste_diario": serie.round(2),
    "media_movil_7d": serie.rolling(7, min_periods=1).mean().round(2),
    "anomalia": es_anomalia.fillna(False),
}).reset_index()
informe.columns = ["fecha", "coste_diario", "media_movil_7d", "anomalia"]

ruta = guardar_csv(informe, "L14_demanda_diaria")
ruta

In [ ]:
# 4. el grafico, con las anomalias marcadas
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(informe["fecha"], informe["coste_diario"], lw=0.7, color="#B0B7C3", label="Coste diario")
ax.plot(informe["fecha"], informe["media_movil_7d"], lw=2, color="#1F4E79", label="Media móvil 7 días")

marcados = informe[informe["anomalia"]]
ax.scatter(marcados["fecha"], marcados["coste_diario"],
           s=28, color="#C0392B", zorder=5, label=f"Anomalías ({len(marcados)})")

ax.set_title("Coste diario de envíos y su tendencia (€)", loc="left", fontsize=12, weight="bold")
ax.set_ylabel("Coste (€)")
ax.grid(axis="y", alpha=0.3)
ax.legend(frameon=False, loc="upper left")
fig.autofmt_xdate()
fig.tight_layout()

rutas_fig = guardar_figura(fig, "L14_demanda_y_anomalias", formatos=("png", "svg"))
plt.show()
resumen_entrega(ruta, rutas_fig)

### Criterios de aceptación

1. `informe` tiene una fila por día del rango, con columnas `fecha`, `coste_diario`,
   `media_movil_7d`, `anomalia` (booleano).
2. `es_anomalia` está calculado sobre las **diferencias**, no sobre los valores.
3. Hay **más de 0 y menos de 5%** de días marcados como anomalía. Si marcas el 30% de
   los días, tu umbral está mal.
4. El gráfico tiene media móvil, serie diaria y anomalías marcadas, y se exporta a
   **PNG y SVG** en `salida/`.
5. Escribes en el informe una frase: ¿cuál fue la anomalía más grande y por qué crees
   que ocurrió?